In [3]:
# !pip install requests beautifulsoup4 lxml -q
# !pip install python-docx


In [4]:
import os
import requests
from bs4 import BeautifulSoup

import json
import pandas as pd
from datetime import datetime
import urllib3
import urllib.parse
import re
import warnings
from urllib.parse import urlencode
import time
import random
from requests.exceptions import RequestException

from pathlib import Path

from collections import Counter
import tqdm

warnings.filterwarnings('ignore', message='Unverified HTTPS request')


In [45]:
# Cấu hình
VBPL_BASE_url = "https://vbpl.vn"
VBPL_SEARCH_url = f"{VBPL_BASE_url}/Pages/timkiem-nangcao.aspx"
SAVE_url_PATH = "./data/raw/collected_vbhd_urls"

PROJECT_ROOT = 'e:\laborlaw-advisor\\'

# url endpoint(Network tab)
SEARCH_url = "https://vbpl.vn/VBQPPL_UserControls/Publishing_22/TimKiem/p_KetQuaTimKiemVanBan.aspx"


#result 
COLLECTED_VBHD_DIR = 'E:\\laborlaw-advisor\\src\\data\\raw\\collected_vbhd'
COLLECTED_VBHD_JSON_PATH = "E:\\laborlaw-advisor\\src\\data\\raw\\collected_vbhd.json"
FAILED_DOWNLOAD_JSON_PATH = 'E:\\laborlaw-advisor\\src\\data\\raw\\collected_vbhd\\failed_download.json'


In [6]:
# Khởi tạo session
def create_session():
    #Tạo session với headers 
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.8',
        'Referer': VBPL_SEARCH_url,
        'Origin': VBPL_BASE_url,
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    })
    return session


def test_create_session():
    session = create_session()

    assert session.headers['Referer'] == VBPL_SEARCH_url
    assert session.headers['Origin'] == VBPL_BASE_url
    assert 'Mozilla' in session.headers['User-Agent']

    print("=> Headers tồn tại và đúng giá trị")

    # test real request
    response = session.get("https://httpbin.org/headers")

    assert response.status_code == 200

    data = response.json()['headers']

    print("Headers gửi đi")
    print(data)

    # kiểm tra header có được gửi ko
    assert 'Mozilla' in data.get('User-Agent', '')
    print("=> Headers đã được gửi đúng")

    #test access
    response = session.get(VBPL_SEARCH_url)

    print("Status:", response.status_code)

    assert response.status_code == 200
    print("=> Truy cập VBPL thành công")


test_create_session()

=> Headers tồn tại và đúng giá trị
Headers gửi đi
{'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Encoding': 'gzip, deflate', 'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.8', 'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8', 'Host': 'httpbin.org', 'Origin': 'https://vbpl.vn', 'Referer': 'https://vbpl.vn/Pages/timkiem-nangcao.aspx', 'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36', 'X-Amzn-Trace-Id': 'Root=1-69d37577-3f5be84e6e789d5402dffff5'}
=> Headers đã được gửi đúng
Status: 200
=> Truy cập VBPL thành công


In [7]:
'''
network -> search:

https://vbpl.vn/VBQPPL_UserControls/Publishing_22/TimKiem/p_KetQuaTimKiemVanBan.aspx?
SearchIn=VBPQFulltext
&DivID=resultSearch
&IsVietNamese=True
&type=1
&s=1
&DonVi=13
&Keyword=lu%E1%BA%ADt%20lao%20%C4%91%E1%BB%99ng&stemp=0
&TimTrong1=Title
&TimTrong1=Title1
&ddrDiaPhuong=99999
&order=VBPQNgayBanHanh
&TypeOfOrder=False
&LoaiVanBan=17,18,20,22
&TrangThaiHieuLuc=2

'''

'\nnetwork -> search:\n\nhttps://vbpl.vn/VBQPPL_UserControls/Publishing_22/TimKiem/p_KetQuaTimKiemVanBan.aspx?\nSearchIn=VBPQFulltext\n&DivID=resultSearch\n&IsVietNamese=True\n&type=1\n&s=1\n&DonVi=13\n&Keyword=lu%E1%BA%ADt%20lao%20%C4%91%E1%BB%99ng&stemp=0\n&TimTrong1=Title\n&TimTrong1=Title1\n&ddrDiaPhuong=99999\n&order=VBPQNgayBanHanh\n&TypeOfOrder=False\n&LoaiVanBan=17,18,20,22\n&TrangThaiHieuLuc=2\n\n'

In [8]:
def save_to_txt(content, file_path):
    full_path = os.path.join(PROJECT_ROOT, file_path)


    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(content)

    print("Lưu tại:", full_path)


def save_to_json(content, file_path):
    full_path = os.path.join(PROJECT_ROOT, file_path)


    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    
    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(content, f, ensure_ascii=False, indent=2)

    print("Lưu tại:", full_path)

In [9]:
def get_total_pages(html):
    #Lấy tổng số trang từ html đầu vào
    soup = BeautifulSoup(html, 'lxml')
    
    last_link = None
    all_links = soup.find_all('a') # lấy all thẻ a 
    for link in all_links:
        text = link.get_text()  # lấy text bên trong thẻ a
        if text is not None:
            if 'Cuối' in text:
                last_link = link
                break # end khi tìm thấy

    if last_link and 'href' in last_link.attrs:
        # Extract số từ javascript:Nexpage('resultSearch','số cần lấy')
        match = re.search(r"'(\d+)'\s*\)", last_link['href'])
        if match:
            return int(match.group(1))
        

    return 1 # nếu ko match thì mặc định là 1 page trang chủ


html_nav_test_string = '''

    <div class="paging">
    <a href="javascript:;" class="current">1</a>
    <a href="javascript:Nexpage('resultSearch','2');">2</a>
    <a href="javascript:Nexpage('resultSearch','3');">3</a>
    <a href="javascript:Nexpage('resultSearch','4');">4</a>
    <span>...</span>

    <a href="javascript:Nexpage('resultSearch','2');">Sau</a>
    <a href="javascript:Nexpage('resultSearch','13');">Cuối »</a>

    </div>

</div>


'''
print('test total page:', get_total_pages(html_nav_test_string))

test total page: 13


In [10]:
''' 
<li class="download">
    <a href="javascript:downloadfile('21.2023.QH15.doc','/TW/Lists/vbpq/Attachments/167658/21.2023.QH15.doc');">
        Tải về
    </a>
</li>


'''

' \n<li class="download">\n    <a href="javascript:downloadfile(\'21.2023.QH15.doc\',\'/TW/Lists/vbpq/Attachments/167658/21.2023.QH15.doc\');">\n        Tải về\n    </a>\n</li>\n\n\n'

In [41]:

def extract_document_info(li_element):
    #Trích xuất thông tin chi tiết từ 1 the li

    info = {
        'title': '',
        'url_fulltext': '',
        'url_pdf': '',
        'url_doc': '',
        'filename_doc': '',
        'item_id': '',
        'ban_hanh': '',
        'hieu_luc': '',
        'mo_ta': ''
    }
    title_link = li_element.select_one('.title a[href*="ItemID="]')
    if title_link:
        info['title'] = title_link.get_text(strip=True)
        info['url_fulltext'] = title_link.get('href', '')
        
        # Trích xuất ItemID
        match = re.search(r'ItemID=(\d+)', info['url_fulltext'])
        if match:
            info['item_id'] = match.group(1)

    # Lấy url pdf bản pdf
    pdf_link = li_element.select_one('li.source a')
    if pdf_link:
        info['url_pdf'] = pdf_link.get('href', '')

    # Lấy url file doc từ link tải về
    download_link = li_element.select_one('li.download a')
    if download_link:
        href = download_link.get('href', '')
        # Parse từ javascript:downloadfile('filename.doc', '/path/to/file.doc')
        match = re.search(r"downloadfile\('([^']+)',\s*'([^']+)'\)", href)
        if match:
            filename = match.group(1)
            filepath = match.group(2)
            info['url_doc'] = filepath
            info['filename_doc'] = filename
    
    # Lấy ngày ban hành và hiệu lực
    right_paragraphs = li_element.select('.right p.green')
    for p in right_paragraphs:
        text = p.get_text(strip=True)
        if 'Ban hành:' in text:
            info['ban_hanh'] = text.replace('Ban hành:', '').strip()
        elif 'Hiệu lực:' in text:
            info['hieu_luc'] = text.replace('Hiệu lực:', '').strip()
    
    # Lấy mô tả
    mo_ta = li_element.select_one('.des p')
    if mo_ta:
        info['mo_ta'] = mo_ta.get_text(strip=True)
    
    return info


def search_vbpl(session, keyword, page=1, trang_thai_hieu_luc=2):  
    params = {
        'SearchIn': 'VBPQFulltext',
        'DivID': 'resultSearch',
        'IsVietNamese': 'True',
        'Page': str(page),
        'type': '1',
        's': '1',
        'DonVi': '13',
        'Keyword': keyword,
        'stemp': '0',
        'TimTrong1': ['Title', 'Title1'],
        'ddrDiaPhuong': '99999',
        'order': 'VBPQNgayBanHanh',
        'TypeOfOrder': 'False',
        'LoaiVanBan': '17,18,20,22',
        'TrangThaiHieuLuc': str(trang_thai_hieu_luc)
    }
    
    response = session.post(SEARCH_url, params=params, timeout=30)
    if response.status_code == 200:
        return response.text
    else:
        raise Exception(f"Request failed: {response.status_code}")

def search_all_pages(keyword, trang_thai_hieu_luc=2, max_pages=None, base_url="https://vbpl.vn"):
    """Tìm kiếm all các trang và trích xuất thông tin chi tiết"""
    session = create_session()
    all_documents = []
    failed_pages = []
    
    print(f"Đang tìm kiếm: {keyword}")
    # Lấy trang 1
    try:
        html = search_vbpl(session, keyword, page=1, trang_thai_hieu_luc=trang_thai_hieu_luc)
    except Exception as e:
        print(f"ko thể tải trang 1: {str(e)}")
        return all_documents, [1]
    
    save_to_txt(html, f'test/html_page_1')
    soup = BeautifulSoup(html, 'lxml')
    
    # Lấy all thẻ li chứa văn bản
    results = soup.select('.listLaw > li')
    
    for li in results:
        doc_info = extract_document_info(li)
        # Tạo url đầy đủ
        if doc_info['url_pdf']:
            doc_info['url_pdf_full'] = base_url + doc_info['url_pdf']
        if doc_info['url_doc']:
            doc_info['url_doc_full'] = base_url + doc_info['url_doc']
        if doc_info['url_fulltext']:
            doc_info['url_fulltext_full'] = base_url + doc_info['url_fulltext']
        
        all_documents.append(doc_info)
    
    print(f"Trang 1: {len(results)} kết quả")
    
    # Tổng số trang
    total_pages = get_total_pages(html)
    print(f"Tổng số trang: {total_pages}")
    
    # Giới hạn số trang
    if max_pages:
        total_pages = min(total_pages, max_pages)
    
    # Lấy các trang còn lại với delay
    for page in range(2, total_pages + 1):
        try:
            # Delay ngẫu nhiên 
            time.sleep(random.uniform(1, 3))
            
            html = search_vbpl(session, keyword, page=page, trang_thai_hieu_luc=trang_thai_hieu_luc)
            save_to_txt(html, f'test/html_page_{page}')
            
            soup = BeautifulSoup(html, 'lxml')
            results = soup.select('.listLaw > li')
            
            for li in results:
                doc_info = extract_document_info(li)
                if doc_info['url_pdf']:
                    doc_info['url_pdf_full'] = base_url + doc_info['url_pdf']
                if doc_info['url_doc']:
                    doc_info['url_doc_full'] = base_url + doc_info['url_doc']
                if doc_info['url_fulltext']:
                    doc_info['url_fulltext_full'] = base_url + doc_info['url_fulltext']
                    
                all_documents.append(doc_info)
            
            print(f"Trang {page}: {len(results)} kết quả")
            
        except Exception as e:
            print(f"Lỗi trang {page}: {str(e)}")
            failed_pages.append(page)
            continue
    
    if failed_pages:
        print(f"\nCác trang bị lỗi: {failed_pages}")
    
    return all_documents, failed_pages


# run 
all_documents, failed_pages = search_all_pages(
    keyword="luật lao động",
    trang_thai_hieu_luc=2
)

print(f"\nTổng cộng: {len(all_documents)} văn bản")
print(f"Số trang lỗi: {len(failed_pages)}")

result_obj = {
    "total": len(all_documents),
    "failed_pages": failed_pages,
    "documents": all_documents
}

save_to_json(result_obj, COLLECTED_VBHD_JSON_PATH)


Đang tìm kiếm: luật lao động
Lưu tại: e:\laborlaw-advisor\test/html_page_1
Trang 1: 30 kết quả
Tổng số trang: 13
Lưu tại: e:\laborlaw-advisor\test/html_page_2
Trang 2: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_3
Trang 3: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_4
Trang 4: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_5
Trang 5: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_6
Trang 6: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_7
Trang 7: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_8
Trang 8: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_9
Trang 9: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_10
Trang 10: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_11
Trang 11: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_12
Trang 12: 30 kết quả
Lưu tại: e:\laborlaw-advisor\test/html_page_13
Trang 13: 23 kết quả

Tổng cộng: 383 văn bản
Số trang lỗi: 0
Lưu tại: E:\laborlaw-advisor\src\data\raw\colle

In [42]:
# test url
test_url = "https://vbpl.vn/TW/Pages/vbpq-van-ban-goc.aspx?ItemID=179106"
response = requests.get(test_url, timeout=30)
soup = BeautifulSoup(response.content, 'html.parser')

# tim link real
pdf_links = soup.find_all('a', href=True)
for link in pdf_links:
    if 'pdf' in link['href'].lower() or 'download' in link['href'].lower():
        print(f"Found: {link['href']}")
        print(f"Text: {link.get_text(strip=True)}")

Found: javascript:ShowDialogDownload();
Text: Tải về
Found: javascript:downloadfile('80_2025_QH15_649688.doc','/TW/Lists/vbpq/Attachments/179106/80_2025_QH15_649688.doc');
Text: 80_2025_QH15_649688.doc
Found: javascript:downloadfile('VanBanGoc_80.2025.QH15.pdf','/FileData/TW/Lists/vbpq/Attachments/179106/VanBanGoc_80.2025.QH15.pdf');
Text: 80.2025.QH15.pdf


In [43]:
def create_download_url(filepath, filename):
    base = "https://vbpl.vn/VBQPPL_UserControls/Publishing_22/pActiontkeFile.aspx"
    params = {
        'do': 'download',
        'urlfile': filepath,  # /TW/Lists/vbpq/Attachments/173625/52.2024.QH15.doc
        'filename': filename   # 52.2024.QH15.doc
    }
    
    # Tạo url với query string
    from urllib.parse import urlencode
    return f"{base}?{urlencode(params)}"

In [47]:
def extract_download_url(aspx_url): 
    """ 
    Trích xuất url download thật từ trang .aspx 
    doc,docx --> pdf 
    """ 
    try: 
        response = requests.get(aspx_url, timeout=30) 
        soup = BeautifulSoup(response.content, 'html.parser')
        # Tìm all link có downloadfile()
        download_links = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            if 'downloadfile' in href:
                # Extract: downloadfile('filename','/path/to/file')
                match = re.search(r"downloadfile\('([^']+)','([^']+)'\)", href)
            if match:
                filename = match.group(1)
                filepath = match.group(2)
                download_links.append({
                    'filename': filename,
                    'path': filepath,
                    'ext': filename.split('.')[-1].lower()
                })
    
        # Ưu tiên doc/docx
        for link in download_links:
            if link['ext'] in ['doc', 'docx']:
                return f"https://vbpl.vn{link['path']}", link['filename']
        
        # Nếu ko có doc, lấy pdf
        for link in download_links:
            if link['ext'] == 'pdf':
                return f"https://vbpl.vn{link['path']}", link['filename']
        
        return None, None
        
    except Exception as e:
        print(f"Lỗi extract url: {e}")
        return None, None


def download_1_document(item, output_dir):
    """
    Tai 1 van ban
    url_doc_full --> url_pdf_full
    """
    item_id = item.get('item_id', 'unknown')
    title = item.get('title', 'untitled').replace('/', '-').replace('\\', '-')
    
    #check file đã tồn tại chưa
    possible_extensions = ['.doc', '.docx', '.pdf']
    for ext in possible_extensions:
        sub_dir = output_dir / ext.replace('.', '')
        filename = f"{item_id}_{title[:50]}{ext}"
        filepath = sub_dir / filename

        if filepath.exists():
            print(f"Đã có: {filename} --> bỏ qua")
            return {
                'item_id': item_id,
                'title': title,
                'filename': filename,
                'filepath': str(filepath),
                'extension': ext,
                'status': 'skipped'  
            }


    # Neu co url_doc_full -> tai truc tiep
    # Neu ko, parse url_pdf_full de tim doc/pdf
    download_url = None
    source_type = None
    
    # check url_doc_full truoc
    if item.get('url_doc_full'):
        download_url = item['url_doc_full']
        source_type = 'direct_doc'
    
    # Neu ko co, parse url_pdf_full
    elif item.get('url_pdf_full'):
        print(f"  --> Parsing ASPX page", end=' ')
        real_url, real_filename = extract_download_url(item['url_pdf_full'])
        if real_url:
            download_url = real_url
            source_type = 'parsed_aspx'
        else:
            print("ko tìm thấy link download")
            return {
                'item_id': item_id,
                'title': title,
                'status': 'failed',
                'error': 'no_download_link_found'
            }
    
    if not download_url:
        print("ko có url để tải")
        return {
            'item_id': item_id,
            'title': title,
            'status': 'failed',
            'error': 'no_url'
        }
    
    # Xac dinh extension
    if download_url.endswith('.docx'):
        ext = '.docx'
    elif download_url.endswith('.doc'):
        ext = '.doc'
    elif download_url.endswith('.pdf'):
        ext = '.pdf'
    else:
        # Fallback: lay tu url
        ext = '.doc'
    # tạo thư mục con 
    if ext == '.doc':
        sub_dir = output_dir / 'doc'
    elif ext == '.docx':
        sub_dir = output_dir / 'docx'
    elif ext == '.pdf':
        sub_dir = output_dir / 'pdf'
    else:
        sub_dir = output_dir / 'other'

    sub_dir.mkdir(exist_ok=True)  # tạo dir nếu chưa có

    filename = f"{item_id}_{title[:50]}{ext}"  # giới hạn tên
    filepath = sub_dir / filename

    
    # tải file
    try:
        response = requests.get(download_url, timeout=30)
        response.raise_for_status()
        
        # Luu file
        with open(filepath, 'wb') as f:
            f.write(response.content)
        
        print(f"Đã lưu: {filename} ({source_type})")
        
        return {
            'item_id': item_id,
            'title': title,
            'filename': filename,
            'filepath': str(filepath),
            'extension': ext,
            'source_type': source_type,
            'download_url': download_url,
            'status': 'success'
        }
    
    except Exception as e:
        print(f"lỗi: {e}")
        return {
            'item_id': item_id,
            'title': title,
            'status': 'failed',
            'error': str(e),
            'download_url': download_url
        }


def download_all_documents(json_file, output_dirname='vbhd', delay=0.5, limit=None):
    """
    tải all văn bản từ json
    limit: số lượng file tải (none = tải all)
    """
    # Doc JSON
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    documents = data.get('documents', [])
    
    # giới hạn số lượng
    if limit:
        documents = documents[:limit]
    
    total = len(documents)
    print(f'Tổng số văn bản: {total}\n')
    
    # tạo thư mục gốc và các thư mục con
    output_dir = Path(output_dirname)
    output_dir.mkdir(exist_ok=True)

    # tạo các thư mục con
    (output_dir / 'doc').mkdir(exist_ok=True)
    (output_dir / 'docx').mkdir(exist_ok=True)
    (output_dir / 'pdf').mkdir(exist_ok=True)

    
    # tiến hành tải
    results = []
    success = 0
    failed = 0
    skipped = 0
    
    for idx, doc in enumerate(documents, 1):
        print(f"[{idx}/{total}] {doc.get('title', 'N/A')[:50]}", end=' ')
        
        result = download_1_document(doc, output_dir)
        results.append(result)
        
        if result and result['status'] == 'success':
            success += 1
        elif result['status'] == 'skipped':
            skipped += 1
        else:
            failed += 1
        
        time.sleep(delay)

  
    print(f"success: {success}/{total} ({success/total*100:.1f}%)")
    print(f"fail: {failed}/{total}")
    print(f"Bỏ qua (đã có): {skipped}/{total}") 
    
    # Thong ke theo loai file (doc, docx, pdf)
    stats = {}
    for r in results:
        if r['status'] == 'success':
            ext = r.get('extension', 'unknown')
            stats[ext] = stats.get(ext, 0) + 1
    
    print(f"\nThong ke theo dinh dang:")
    for ext, count in stats.items():
        print(f"  {ext}: {count} file")
    
    # lưu log
    log_file = output_dir / 'download_log.json'
    with open(log_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\nLog đã lưu: {log_file}")
    
    # lưu list thất bại
    failed_items = [r for r in results if r['status'] == 'failed']
    if failed_items:
        failed_file = output_dir / 'failed_download.json'
        with open(failed_file, 'w', encoding='utf-8') as f:
            json.dump(failed_items, f, ensure_ascii=False, indent=2)
        print(f"list thất bại: {failed_file}")
    
    return results

#result 
results = download_all_documents( 
    json_file=COLLECTED_VBHD_JSON_PATH, 
    output_dirname=COLLECTED_VBHD_DIR,
    delay=0.5
    )

Tổng số văn bản: 383

[1/383] Luật 80/2025/QH15 Đã có: 179106_Luật 80-2025-QH15.doc --> bỏ qua
[2/383] Luật 73/2025/QH15 Đã có: 179262_Luật 73-2025-QH15.docx --> bỏ qua
[3/383] Luật 74/2025/QH15 Đã có: 179273_Luật 74-2025-QH15.doc --> bỏ qua
[4/383] Luật 71/2025/QH15 Đã có: 179989_Luật 71-2025-QH15.docx --> bỏ qua
[5/383] Luật 52/2024/QH15 Đã có: 173625_Luật 52-2024-QH15.doc --> bỏ qua
[6/383] Luật Luật số 51/2024/QH15 Đã có: 172923_Luật Luật số 51-2024-QH15.doc --> bỏ qua
[7/383] Luật 36/2024/QH15 Đã có: 170620_Luật 36-2024-QH15.doc --> bỏ qua
[8/383] Luật 21/2023/QH15 Đã có: 167658_Luật 21-2023-QH15.doc --> bỏ qua
[9/383] Luật 69/2020/QH14 Đã có: 146643_Luật 69-2020-QH14.doc --> bỏ qua
[10/383] Luật 99/2015/QH13 Đã có: 96121_Luật 99-2015-QH13.doc --> bỏ qua
[11/383] Luật 47/2014/QH13 Đã có: 36823_Luật 47-2014-QH13.doc --> bỏ qua
[12/383] Luật 53/2010/QH12 Đã có: 25681_Luật 53-2010-QH12.doc --> bỏ qua
[13/383] Luật 73/2006/QH11 Đã có: 14838_Luật 73-2006-QH11.doc --> bỏ qua
[14/383] Ng

In [48]:
# Đọc file failed_download.json

with open(FAILED_DOWNLOAD_JSON_PATH, 'r', encoding='utf-8') as f:
    failed = json.load(f)

# Xem các lỗi
for item in failed[:50]:  
    print(f"ID: {item['item_id']}")
    print(f"Title: {item['title']}")
    print(f"Lỗi: {item['error']}")


ID: 12434
Title: Nghị quyết 01-NQ-CP
Lỗi: no_url
ID: 169619
Title: Nghị định 97-2022-NĐ-CP
Lỗi: no_download_link_found
ID: 169627
Title: Nghị định 88-2022-NĐ-CP
Lỗi: no_download_link_found
ID: 158798
Title: Nghị định 35-2022-NĐ-CP
Lỗi: no_download_link_found
ID: 143476
Title: Nghị định 84-2020-NĐ-CP
Lỗi: no_download_link_found
ID: 25074
Title: Nghị định 29-2010-NĐ-CP
Lỗi: no_url
ID: 13759
Title: Nghị định 126-2007-NĐ-CP
Lỗi: no_url
ID: 13941
Title: Nghị định 110-2007-NĐ-CP
Lỗi: no_url
ID: 174097
Title: Thông tư 16-2024-TT-BVHTTDL
Lỗi: no_url
ID: 170282
Title: Thông tư 07-2024-TT-BVHTTDL
Lỗi: no_url
ID: 153297
Title: Thông tư 18-2021-TT-BLĐTBXH
Lỗi: no_download_link_found
ID: 138489
Title: Thông tư 39-2019-TT-BGTVT
Lỗi: no_url
ID: 118831
Title: Thông tư 14-2016-TT-BCA
Lỗi: no_url
ID: 60295
Title: Thông tư 04-2015-TT-BLĐTBXH
Lỗi: no_url
ID: 24872
Title: Thông tư 42-2009-TT-BLĐTBXH
Lỗi: no_url
ID: 12061
Title: Thông tư 19-2009-TT-BQP
Lỗi: no_url
ID: 26056
Title: Thông tư 21-2007-TT-BLĐTBX

In [49]:
# Đếm các loại lỗi
error_types = Counter(item['error'].split(':')[0] for item in failed)
for error, count in error_types.items():
    print(f"{error}: {count} files")

no_url: 21 files
no_download_link_found: 5 files


In [50]:
def categorize_errors(failed_docs):
    #lọc lỗi server để tải lại 

    server_error = []   # 5..
    skip = []  
    for doc in failed_docs:
        error = doc['error'].lower()
        if '500' in error or '503' in error or '502' in error or 'timeout' in error:
            server_error.append(doc)
        else:
            skip.append(doc)
    
    return {
        'server_error': server_error,
        'skip': skip
    }

categorized_failed = categorize_errors(failed) # dictionary 

print('timeout: ',len(categorized_failed['server_error']))
print('skip: ',len(categorized_failed['skip']))

timeout:  0
skip:  26


In [51]:
def retry_download(
    failed_json: str,
    output_dirname: str,
    retry_server_errors: bool = True,
    max_retries: int = 3,
    delay: float = 2.0
):  
    with open(failed_json, 'r', encoding='utf-8') as f:
        failed_docs = json.load(f)
    categorized = categorize_errors(failed_docs)
    to_retry = []
    if retry_server_errors:
        to_retry.extend(categorized.get('server_error', []))

    if not to_retry:
        print('k có file nào cần tải lại')

    results = {
        'success': [],
        'failed': [],
        'skipped': categorized.get('skip', [])
    }

    for doc in to_retry:
        url = doc.get('download_url') #"download_url": "https://vbpl.vn/TW/Lists/
        if not url:
            results['skipped'].append(doc)
            continue
        
        success = False
        last_error = None
        
        # Tạo filename từ url
        filename = url.split('/')[-1]
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, timeout=60)
                response.raise_for_status()
                
                filepath = os.path.join(output_dirname, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                results['success'].append({
                    'doc_id': doc['item_id'],
                    'title': doc['title'],
                    'filename': filename,
                    'retry_attempt': attempt + 1
                })
                success = True
                break
                
            except Exception as e:
                last_error = str(e)
                if attempt < max_retries - 1:
                    time.sleep(delay * (attempt + 1))
        
        if not success:
            results['failed'].append({
                'doc_id': doc['item_id'],
                'title': doc['title'],
                'url': url,
                'filename': filename,
                'error': last_error
            })
        
        time.sleep(delay)
    
    # lưu kết quả
    retry_log = os.path.join(output_dirname, 'retry_log.json')
    with open(retry_log, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"\nRetry thành công: {len(results['success'])}")
    print(f"thất bại: {len(results['failed'])}")
    print(f"bỏ qua: {len(results['skipped'])}")
    
    return results



results = retry_download(
    failed_json=FAILED_DOWNLOAD_JSON_PATH,
    output_dirname= COLLECTED_VBHD_DIR,
    retry_server_errors=True,
    max_retries=3,
    delay=2.0
)


k có file nào cần tải lại

Retry thành công: 0
thất bại: 0
bỏ qua: 26
